# Enrichment & Filtering

Part of the [PyMAUDE](../) examples — see [`quickstart.ipynb`](../quickstart.ipynb) first if you haven't loaded a database yet.

This notebook covers joining extra context onto a search result — patient outcomes, device problem codes, patient problem codes — and then narrowing results down with the matching `filter_by_*` methods.

---
## Contents
1. [Setup](#1-setup)
2. [Enrich with patient outcomes](#2-patient)
3. [Enrich with problem codes](#3-problems)
4. [Chained filters](#4-filters)

---
## 1. Setup <a id="1-setup"></a>

In [ ]:
from pymaude import MaudeDatabase

DB_PATH  = '../maude.duckdb'
DATA_DIR = '../maude_data'
YEARS    = '2024-2026'

db = MaudeDatabase(DB_PATH, data_dir=DATA_DIR, verbose=True, memory_limit='2GB')
db.add_years(
    YEARS,
    tables=['master', 'device', 'text', 'patient', 'device_problem', 'patient_problem'],
    download=False
)

Note: `master`, `device_problem`, and `patient_problem` are cumulative tables — they're fully replaced on every reload rather than updated year-by-year, so `add_years()` rejects a request that doesn't cover every year they already have loaded (it would otherwise silently drop the missing years). Pass `force_partial=True` if you really do want to narrow an existing cumulative table's coverage.

In [ ]:
# Running example dataset for the rest of this notebook
thrombectomy_broad = db.search_by_device_names(
    [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy']
)
print(f'Broad rotational thrombectomy search: {len(thrombectomy_broad):,}')

---
## 2. Enrich with patient outcomes <a id="2-patient"></a>

`enrich_with_patient_data()` left-joins the patient outcomes table. `SEQUENCE_NUMBER_OUTCOME` codes:
- `D` Death · `L` Life Threatening · `H` Hospitalization · `S` Disability · `C` Congenital Anomaly · `R` Required Intervention · `O` Other · `U` Unknown · `I` No Information · `A` Not Applicable · `*` Invalid Data

A single patient record can carry more than one code at once, e.g. `"H; O"` — so don't match this field with `==`.

In [ ]:
enriched = db.enrich_with_patient_data(thrombectomy_broad)

with_outcome = enriched.dropna(subset=['SEQUENCE_NUMBER_OUTCOME'])
print(f'Events with patient outcome data: {len(with_outcome):,} / {len(enriched):,}')
with_outcome[['MDR_REPORT_KEY', 'BRAND_NAME', 'SEQUENCE_NUMBER_OUTCOME']].head(10)

In [ ]:
OUTCOME_LABELS = {
    'D': 'Death', 'L': 'Life Threatening', 'H': 'Hospitalization',
    'S': 'Disability', 'C': 'Congenital Anomaly', 'R': 'Required Intervention',
    'O': 'Other', 'U': 'Unknown', 'I': 'No Information', 'A': 'Not Applicable',
    '*': 'Invalid Data',
}

outcome_counts = (
    with_outcome['SEQUENCE_NUMBER_OUTCOME']
    .str.split(';')
    .explode()
    .str.strip()
    .map(lambda c: OUTCOME_LABELS.get(c, c))
    .value_counts()
)
outcome_counts

---
## 3. Enrich with problem codes <a id="3-problems"></a>

MAUDE codes adverse events from two angles: `enrich_with_device_problems()` left-joins the device problem code table (`device_problem`), and `enrich_with_patient_problems()` left-joins the patient problem code table (`patient_problem`). Per `TABLE_METADATA`, FDA's documented start year is 1993 for both — but neither table has a per-row date column, so actual coverage isn't independently verified by this library; see `db.info()`.

In [ ]:
with_problems = db.enrich_with_device_problems(thrombectomy_broad)

problem_counts = (
    with_problems
    .dropna(subset=['DEVICE_PROBLEM_CODE'])
    ['DEVICE_PROBLEM_CODE']
    .value_counts()
)
print(f'Events with a device problem code: {problem_counts.sum():,}')
print('\nTop device problem codes:')
problem_counts.head(10)

In [ ]:
with_patient_problems = db.enrich_with_patient_problems(thrombectomy_broad)

patient_problem_counts = (
    with_patient_problems
    .dropna(subset=['PATIENT_PROBLEM_CODE'])
    ['PATIENT_PROBLEM_CODE']
    .value_counts()
)
print(f'Events with a patient problem code: {patient_problem_counts.sum():,}')
print('\nTop patient problem codes:')
patient_problem_counts.head(10)

---
## 4. Chained filters <a id="4-filters"></a>

The `filter_by_*` methods take a results DataFrame and return a narrowed-down copy — there's no query-builder object, just plain DataFrames passed from one call to the next.

| Method | Requires first | Matches on |
|---|---|---|
| `filter_by_outcome(df, code)` | `enrich_with_patient_data()` | `SEQUENCE_NUMBER_OUTCOME` — membership, since one record can carry multiple codes (e.g. `"H; O"`) |
| `filter_by_patient(df, age_min=, age_max=, sex=)` | `enrich_with_patient_data()` | `PATIENT_AGE` / `PATIENT_SEX` |
| `filter_by_device_problem(df, code)` | `enrich_with_device_problems()` | `DEVICE_PROBLEM_CODE` |
| `filter_by_patient_problem(df, code)` | `enrich_with_patient_problems()` | `PATIENT_PROBLEM_CODE` |
| `filter_by_narrative(df, term)` | nothing — queries `text` directly | `FOI_TEXT` substring |

Each raises a clear `ValueError` if the required enrich step hasn't been run yet.

In [ ]:
# Deaths among the thrombectomy events (reusing `enriched` from section 2)
deaths = db.filter_by_outcome(enriched, 'D')
print(f'Deaths: {len(deaths):,} / {len(enriched):,} enriched records')
deaths[['MDR_REPORT_KEY', 'BRAND_NAME', 'SEQUENCE_NUMBER_OUTCOME']].head(10)

In [ ]:
# Demographic filters — age range and/or sex
elderly = db.filter_by_patient(enriched, age_min=65)
print(f'Events in patients 65+: {len(elderly):,}')

females = db.filter_by_patient(enriched, sex='Female')
print(f'Events in female patients: {len(females):,}')

In [ ]:
# Filter to a specific device problem code (reusing `with_problems`/`problem_counts` from section 3)
top_code = problem_counts.index[0]
events_with_top_code = db.filter_by_device_problem(with_problems, top_code)
print(f'Events with device problem code {top_code!r}: {len(events_with_top_code):,}')
events_with_top_code[['MDR_REPORT_KEY', 'BRAND_NAME', 'DEVICE_PROBLEM_CODE']].head(10)

In [ ]:
# Filter to a specific patient problem code (reusing `with_patient_problems`/`patient_problem_counts` from section 3)
if len(patient_problem_counts) > 0:
    top_patient_code = patient_problem_counts.index[0]
    events_with_top_patient_code = db.filter_by_patient_problem(with_patient_problems, top_patient_code)
    print(f'Events with patient problem code {top_patient_code!r}: {len(events_with_top_patient_code):,}')
    events_with_top_patient_code[['MDR_REPORT_KEY', 'BRAND_NAME', 'PATIENT_PROBLEM_CODE']].head(10)

In [ ]:
# Narrative keyword filter — no enrich step required, queries `text` directly
battery_related = db.filter_by_narrative(thrombectomy_broad, 'battery')
print(f'Narratives mentioning "battery": {len(battery_related):,}')
battery_related[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME']].head(10)

In [ ]:
# Chaining several filters: device search → enrich → outcome → demographic
step1 = db.enrich_with_patient_data(thrombectomy_broad)
step2 = db.filter_by_outcome(step1, 'D')
fatal_over50 = db.filter_by_patient(step2, age_min=50)

print(f'Fatal thrombectomy events in patients 50+: {len(fatal_over50):,}')
fatal_over50[['MDR_REPORT_KEY', 'BRAND_NAME', 'PATIENT_AGE', 'SEQUENCE_NUMBER_OUTCOME']]

In [ ]:
db.close()